# SINGER-Inspired Spatial Local-Subgraph Discovery

This notebook scans the complete chromosome spatially, builds informative low-recombination windows, and extracts each selected window's original local ARG subgraph with fixed interfaces. It is designed to prepare later coalescence/recombination proposals while leaving all material outside the selected half-open interval unchanged.

The workflow is inspired by [SINGER's spatial subgraph strategy](https://www.nature.com/articles/s41588-025-02317-9), but it does **not** implement an HMM, MCMC sampler, badness score, or replacement actions.


In [ ]:
from __future__ import annotations

import gc
import json
import os
from pathlib import Path
import resource
import sys
import threading
import time

from IPython.display import display
from numba import njit
import numpy as np
import tskit


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start).expanduser().resolve()
    fallback = Path("/Users/pratik/Documents/work/aim3/simpliied")
    for path in (start, *start.parents, fallback):
        if (path / "arg/new_rl/trace.py").is_file() and (
            path / "argscape/synthetic_full_arg.py"
        ).is_file():
            return path
    raise RuntimeError("Could not locate the workspace containing arg/new_rl")


WORKSPACE_ROOT = find_workspace_root()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

import arg.new_rl as new_rl_module
from arg.new_rl import build_fast_trace_from_full_arg
from arg.new_rl.trace import (
    EVENT_KIND_COALESCENCE,
    EVENT_KIND_RECOMBINATION,
)
from argscape import NODE_IS_RE_EVENT, build_synthetic_full_arg

EXPECTED_NEW_RL = (WORKSPACE_ROOT / "arg/new_rl/__init__.py").resolve()
RESOLVED_NEW_RL = Path(new_rl_module.__file__).resolve()
if RESOLVED_NEW_RL != EXPECTED_NEW_RL:
    raise RuntimeError(f"Expected {EXPECTED_NEW_RL}, imported {RESOLVED_NEW_RL}")

print(f"Python: {sys.executable}", flush=True)
print(f"new_rl: {RESOLVED_NEW_RL}", flush=True)


## Configuration


In [ ]:
def environment_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return value.strip().lower() in {"1", "true", "yes", "on"}


DEFAULT_SYNTHETIC_TREE_PATH = Path(
    "/Users/pratik/.lorax/projects/1000Genomes/"
    "1kg_chr2_synthetic_full_arg.trees"
)
DEFAULT_SOURCE_TREE_PATH = Path(
    "/Users/pratik/.lorax/projects/1000Genomes/1kg_chr2.trees.tsz"
)

SYNTHETIC_TREE_PATH = Path(
    os.environ.get("ARG_SYNTHETIC_TREES_PATH", DEFAULT_SYNTHETIC_TREE_PATH)
).expanduser()
SOURCE_TREE_PATH = Path(
    os.environ.get("ARG_SOURCE_TREES_PATH", DEFAULT_SOURCE_TREE_PATH)
).expanduser()
ALLOW_SYNTHETIC_BUILD = environment_flag("ARG_ALLOW_SYNTHETIC_BUILD", False)
SAVE_BUILT_SYNTHETIC = environment_flag("ARG_SAVE_BUILT_SYNTHETIC", False)
REQUIRE_UNIQUE_EVENT_TIMES = environment_flag(
    "ARG_REQUIRE_UNIQUE_EVENT_TIMES",
    True,
)

LOCAL_TARGET_SITES = int(os.environ.get("ARG_LOCAL_TARGET_SITES", "100"))
LOCAL_LOW_RECOMB_QUANTILE = float(
    os.environ.get("ARG_LOCAL_LOW_RECOMB_QUANTILE", "0.10")
)
LOCAL_MAX_WINDOWS = int(os.environ.get("ARG_LOCAL_MAX_WINDOWS", "100"))
LOCAL_BATCH_OFFSET = int(os.environ.get("ARG_LOCAL_BATCH_OFFSET", "0"))
LOCAL_MAX_DISPLAY = int(os.environ.get("ARG_LOCAL_MAX_DISPLAY", "25"))
LOCAL_MAX_TREES_PER_WINDOW = int(
    os.environ.get("ARG_LOCAL_MAX_TREES_PER_WINDOW", "5000")
)
LOCAL_MAX_EDGE_REFERENCES = int(
    os.environ.get("ARG_LOCAL_MAX_EDGE_REFERENCES", "5000000")
)
HEARTBEAT_SECONDS = float(os.environ.get("ARG_HEARTBEAT_SECONDS", "30"))

if LOCAL_TARGET_SITES <= 0:
    raise ValueError("ARG_LOCAL_TARGET_SITES must be positive")
if not 0.0 < LOCAL_LOW_RECOMB_QUANTILE <= 1.0:
    raise ValueError("ARG_LOCAL_LOW_RECOMB_QUANTILE must be in (0, 1]")
if LOCAL_MAX_WINDOWS <= 0:
    raise ValueError("ARG_LOCAL_MAX_WINDOWS must be positive")
if LOCAL_BATCH_OFFSET < 0:
    raise ValueError("ARG_LOCAL_BATCH_OFFSET must be nonnegative")
if LOCAL_MAX_DISPLAY < 0:
    raise ValueError("ARG_LOCAL_MAX_DISPLAY must be nonnegative")
if LOCAL_MAX_TREES_PER_WINDOW <= 0:
    raise ValueError("ARG_LOCAL_MAX_TREES_PER_WINDOW must be positive")
if LOCAL_MAX_EDGE_REFERENCES <= 0:
    raise ValueError("ARG_LOCAL_MAX_EDGE_REFERENCES must be positive")
if HEARTBEAT_SECONDS < 0:
    raise ValueError("ARG_HEARTBEAT_SECONDS must be nonnegative")

configuration = {
    "synthetic_tree_path": str(SYNTHETIC_TREE_PATH),
    "source_tree_path": str(SOURCE_TREE_PATH),
    "allow_synthetic_build": ALLOW_SYNTHETIC_BUILD,
    "save_built_synthetic": SAVE_BUILT_SYNTHETIC,
    "require_unique_event_times": REQUIRE_UNIQUE_EVENT_TIMES,
    "target_sites_per_window": LOCAL_TARGET_SITES,
    "low_recombination_quantile": LOCAL_LOW_RECOMB_QUANTILE,
    "max_deep_trace_windows": LOCAL_MAX_WINDOWS,
    "batch_offset": LOCAL_BATCH_OFFSET,
    "max_display": LOCAL_MAX_DISPLAY,
    "max_trees_per_window": LOCAL_MAX_TREES_PER_WINDOW,
    "max_edge_references_per_window": LOCAL_MAX_EDGE_REFERENCES,
}
configuration


In [ ]:
def emit(message):
    print(message, flush=True)


def max_rss_gib():
    rss = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    if sys.platform == "darwin":
        return rss / 1024**3
    return rss / 1024**2


def run_stage(label, operation):
    started = time.perf_counter()
    stopped = threading.Event()
    succeeded = False

    def report_heartbeat():
        while not stopped.wait(HEARTBEAT_SECONDS):
            elapsed = time.perf_counter() - started
            emit(
                f"{label}: running | elapsed={elapsed:.1f}s | "
                f"max_rss={max_rss_gib():.2f} GiB"
            )

    emit(f"{label}: started | max_rss={max_rss_gib():.2f} GiB")
    reporter = None
    if HEARTBEAT_SECONDS > 0:
        reporter = threading.Thread(target=report_heartbeat, daemon=True)
        reporter.start()
    try:
        result = operation()
        succeeded = True
        return result
    finally:
        stopped.set()
        if reporter is not None:
            reporter.join()
        elapsed = time.perf_counter() - started
        status = "finished" if succeeded else "failed"
        emit(
            f"{label}: {status} | elapsed={elapsed:.1f}s | "
            f"max_rss={max_rss_gib():.2f} GiB"
        )


def load_tree_sequence(path):
    path = Path(path)
    if path.suffix == ".tsz":
        import tszip

        return tszip.decompress(path)
    return tskit.load(str(path))


def tree_sequence_signature(ts):
    return {
        "sequence_length": float(ts.sequence_length),
        "trees": int(ts.num_trees),
        "samples": int(ts.num_samples),
        "nodes": int(ts.num_nodes),
        "edges": int(ts.num_edges),
        "sites": int(ts.num_sites),
        "mutations": int(ts.num_mutations),
    }


def readonly(values, dtype=None):
    array = np.asarray(values, dtype=dtype)
    array.setflags(write=False)
    return array


## Load the Prebuilt Synthetic ARG and Trace


In [ ]:
def synthetic_build_provenance(ts):
    for provenance in reversed(list(ts.provenances())):
        try:
            record = json.loads(provenance.record)
        except (TypeError, ValueError):
            continue
        if record.get("software", {}).get("name") == (
            "local_argscape_synthetic_full_arg"
        ):
            return record
    return None


def load_or_build_synthetic_arg():
    if SYNTHETIC_TREE_PATH.is_file():
        loaded = load_tree_sequence(SYNTHETIC_TREE_PATH)
        flags = np.asarray(loaded.nodes_flags, dtype=np.uint32)
        recombination_nodes = np.flatnonzero((flags & NODE_IS_RE_EVENT) != 0)
        if recombination_nodes.size:
            if recombination_nodes.size % 2:
                raise ValueError("Synthetic ARG has an odd recombination-node count")
            provenance = synthetic_build_provenance(loaded)
            certified_unique = bool(
                provenance
                and provenance.get("parameters", {}).get(
                    "ensure_unique_event_times",
                    False,
                )
                and provenance.get("summary", {}).get(
                    "event_times_are_globally_unique",
                    False,
                )
            )
            if REQUIRE_UNIQUE_EVENT_TIMES and not certified_unique:
                raise ValueError(
                    f"{SYNTHETIC_TREE_PATH} is not certified to have "
                    "globally unique event times; rebuild it with the "
                    "current synthetic ARG converter"
                )
            return loaded, {
                "source": "prebuilt_synthetic_full_arg",
                "path": str(SYNTHETIC_TREE_PATH),
                "recombination_node_count": int(recombination_nodes.size),
                "unique_event_times_certified": certified_unique,
            }
        if not ALLOW_SYNTHETIC_BUILD:
            raise ValueError(
                f"{SYNTHETIC_TREE_PATH} has no explicit recombination nodes; "
                "set ARG_ALLOW_SYNTHETIC_BUILD=1 to convert it"
            )
        source = loaded
        source_path = SYNTHETIC_TREE_PATH
    else:
        if not ALLOW_SYNTHETIC_BUILD:
            raise FileNotFoundError(
                f"Prebuilt synthetic ARG not found: {SYNTHETIC_TREE_PATH}. "
                "Set ARG_ALLOW_SYNTHETIC_BUILD=1 for the expensive conversion."
            )
        if not SOURCE_TREE_PATH.is_file():
            raise FileNotFoundError(SOURCE_TREE_PATH)
        source = load_tree_sequence(SOURCE_TREE_PATH)
        source_path = SOURCE_TREE_PATH

    result = build_synthetic_full_arg(
        source,
        split_rule="balanced",
        ensure_unique_event_times=REQUIRE_UNIQUE_EVENT_TIMES,
    )
    synthetic = result.tree_sequence
    if SAVE_BUILT_SYNTHETIC:
        SYNTHETIC_TREE_PATH.parent.mkdir(parents=True, exist_ok=True)
        synthetic.dump(str(SYNTHETIC_TREE_PATH))
    metadata = dict(result.metadata)
    metadata.update(
        {
            "source": "built_in_notebook",
            "source_path": str(source_path),
            "saved": bool(SAVE_BUILT_SYNTHETIC),
        }
    )
    return synthetic, metadata


synthetic_arg, synthetic_metadata = run_stage(
    "Load or build synthetic full ARG",
    load_or_build_synthetic_arg,
)
synthetic_signature_before = tree_sequence_signature(synthetic_arg)

trace = run_stage(
    "Build FastARGTrace",
    lambda: build_fast_trace_from_full_arg(
        synthetic_arg,
        require_unique_event_times=REQUIRE_UNIQUE_EVENT_TIMES,
    ),
)
_trace_time_differences = np.diff(trace.event_time)
if REQUIRE_UNIQUE_EVENT_TIMES:
    assert np.all(_trace_time_differences > 0.0)
else:
    assert np.all(_trace_time_differences >= 0.0)
trace_signature_before = {
    "steps": int(trace.num_steps),
    "events": int(trace.event_count),
    "nodes": int(trace.node_time.size),
    "edges": int(trace.edge_parent.size),
    "samples": int(trace.sample_nodes.size),
    "event_times_strictly_increasing": bool(
        np.all(_trace_time_differences > 0.0)
    ),
    "adjacent_tied_event_times": int(
        np.sum(_trace_time_differences == 0.0)
    ),
}

assert trace.node_time.size == synthetic_arg.num_nodes
assert trace.edge_parent.size == synthetic_arg.num_edges
assert trace.event_count == trace.num_steps

display(
    {
        "synthetic_arg": synthetic_signature_before,
        "synthetic_metadata": synthetic_metadata,
        "trace": trace_signature_before,
        "max_rss_gib": max_rss_gib(),
    }
)


## Whole-Chromosome Spatial Catalogs


In [ ]:
def build_tree_interval_catalog(ts):
    breakpoints = readonly(ts.breakpoints(as_array=True), np.float64)
    left = readonly(breakpoints[:-1], np.float64)
    right = readonly(breakpoints[1:], np.float64)
    span = readonly(right - left, np.float64)
    tree_index = readonly(np.arange(ts.num_trees, dtype=np.int32))

    site_positions = np.asarray(ts.sites_position, dtype=np.float64)
    site_index_at_breakpoint = np.searchsorted(
        site_positions,
        breakpoints,
        side="left",
    ).astype(np.int64)
    site_count = readonly(np.diff(site_index_at_breakpoint), np.int64)

    if ts.num_sites:
        mutation_site = np.asarray(ts.mutations_site, dtype=np.int64)
        mutations_per_site = np.bincount(
            mutation_site,
            minlength=ts.num_sites,
        ).astype(np.int64)
        mutation_prefix = np.empty(ts.num_sites + 1, dtype=np.int64)
        mutation_prefix[0] = 0
        np.cumsum(mutations_per_site, out=mutation_prefix[1:])
        mutation_at_breakpoint = mutation_prefix[site_index_at_breakpoint]
        mutation_count = readonly(np.diff(mutation_at_breakpoint), np.int64)
    else:
        mutation_count = readonly(np.zeros(ts.num_trees, dtype=np.int64))

    return {
        "tree_index": tree_index,
        "left": left,
        "right": right,
        "span": span,
        "site_count": site_count,
        "mutation_count": mutation_count,
        "breakpoints": breakpoints,
    }


@njit
def _target_site_window_bounds(site_count, target_sites):
    interval_count = site_count.size
    starts = np.empty(interval_count, dtype=np.int32)
    ends = np.empty(interval_count, dtype=np.int32)
    window_count = 0
    start = 0
    accumulated_sites = 0

    for interval_index in range(interval_count):
        accumulated_sites += site_count[interval_index]
        if accumulated_sites >= target_sites:
            starts[window_count] = start
            ends[window_count] = interval_index + 1
            window_count += 1
            start = interval_index + 1
            accumulated_sites = 0

    if start < interval_count:
        starts[window_count] = start
        ends[window_count] = interval_count
        window_count += 1

    return starts[:window_count], ends[:window_count]


def build_local_window_catalog(interval_catalog, target_sites):
    starts, ends = _target_site_window_bounds(
        np.asarray(interval_catalog["site_count"], dtype=np.int64),
        int(target_sites),
    )
    interval_site_prefix = np.empty(
        interval_catalog["site_count"].size + 1,
        dtype=np.int64,
    )
    interval_site_prefix[0] = 0
    np.cumsum(interval_catalog["site_count"], out=interval_site_prefix[1:])

    interval_mutation_prefix = np.empty(
        interval_catalog["mutation_count"].size + 1,
        dtype=np.int64,
    )
    interval_mutation_prefix[0] = 0
    np.cumsum(
        interval_catalog["mutation_count"],
        out=interval_mutation_prefix[1:],
    )

    left = interval_catalog["left"][starts]
    right = interval_catalog["right"][ends - 1]
    span = right - left
    site_count = interval_site_prefix[ends] - interval_site_prefix[starts]
    mutation_count = (
        interval_mutation_prefix[ends] - interval_mutation_prefix[starts]
    )
    tree_count = ends.astype(np.int64) - starts.astype(np.int64)
    tree_change_count = np.maximum(tree_count - 1, 0)
    changes_per_site = np.full(starts.size, np.inf, dtype=np.float64)
    nonzero_sites = site_count > 0
    changes_per_site[nonzero_sites] = (
        tree_change_count[nonzero_sites] / site_count[nonzero_sites]
    )

    return {
        "window_index": readonly(np.arange(starts.size, dtype=np.int32)),
        "start_tree_index": readonly(starts, np.int32),
        "end_tree_index": readonly(ends, np.int32),
        "left": readonly(left, np.float64),
        "right": readonly(right, np.float64),
        "span": readonly(span, np.float64),
        "site_count": readonly(site_count, np.int64),
        "mutation_count": readonly(mutation_count, np.int64),
        "tree_count": readonly(tree_count, np.int64),
        "tree_change_count": readonly(tree_change_count, np.int64),
        "tree_changes_per_site": readonly(changes_per_site, np.float64),
        "target_met": readonly(site_count >= int(target_sites), np.bool_),
    }


def window_record(window_catalog, window_index, selection_rank=None):
    window_index = int(window_index)
    record = {
        "window_id": f"window-{window_index:06d}",
        "window_index": window_index,
        "window_key": (
            float(window_catalog["left"][window_index]),
            float(window_catalog["right"][window_index]),
        ),
        "left": float(window_catalog["left"][window_index]),
        "right": float(window_catalog["right"][window_index]),
        "span": float(window_catalog["span"][window_index]),
        "site_count": int(window_catalog["site_count"][window_index]),
        "mutation_count": int(window_catalog["mutation_count"][window_index]),
        "start_tree_index": int(
            window_catalog["start_tree_index"][window_index]
        ),
        "end_tree_index": int(window_catalog["end_tree_index"][window_index]),
        "tree_count": int(window_catalog["tree_count"][window_index]),
        "tree_change_count": int(
            window_catalog["tree_change_count"][window_index]
        ),
        "tree_changes_per_site": float(
            window_catalog["tree_changes_per_site"][window_index]
        ),
        "target_met": bool(window_catalog["target_met"][window_index]),
    }
    if selection_rank is not None:
        record["selection_rank"] = int(selection_rank)
    return record


def select_low_recombination_windows(
    window_catalog,
    quantile,
    max_windows,
    batch_offset=0,
):
    score = window_catalog["tree_changes_per_site"]
    eligible = window_catalog["target_met"] & np.isfinite(score)
    eligible_indices = np.flatnonzero(eligible)
    if eligible_indices.size == 0:
        raise RuntimeError("No site-complete windows are available")

    threshold = float(np.quantile(score[eligible_indices], float(quantile)))
    low_indices = eligible_indices[score[eligible_indices] <= threshold]
    order = np.lexsort(
        (
            window_catalog["left"][low_indices],
            -window_catalog["span"][low_indices],
            score[low_indices],
        )
    )
    ranked_indices = low_indices[order]
    batch_offset = int(batch_offset)
    selected_indices = ranked_indices[
        batch_offset : batch_offset + int(max_windows)
    ]
    rank_lookup = {
        int(window_index): rank
        for rank, window_index in enumerate(ranked_indices)
    }
    selected = [
        window_record(
            window_catalog,
            int(window_index),
            rank_lookup[int(window_index)],
        )
        for window_index in selected_indices
    ]
    return threshold, readonly(ranked_indices, np.int64), selected


In [ ]:
tree_interval_catalog = run_stage(
    "Build whole-chromosome tree-interval catalog",
    lambda: build_tree_interval_catalog(synthetic_arg),
)
local_window_catalog = run_stage(
    "Build tree-aligned informative windows",
    lambda: build_local_window_catalog(
        tree_interval_catalog,
        LOCAL_TARGET_SITES,
    ),
)
(
    low_recombination_threshold,
    ranked_low_recombination_window_indices,
    selected_local_windows,
) = select_low_recombination_windows(
    local_window_catalog,
    LOCAL_LOW_RECOMB_QUANTILE,
    LOCAL_MAX_WINDOWS,
    LOCAL_BATCH_OFFSET,
)

_interval_long = tree_interval_catalog["span"] >= 5000.0
_interval_long_site_counts = tree_interval_catalog["site_count"][_interval_long]
display(
    {
        "tree_interval_count": int(tree_interval_catalog["left"].size),
        "window_count": int(local_window_catalog["left"].size),
        "target_sites": LOCAL_TARGET_SITES,
        "site_complete_windows": int(np.sum(local_window_catalog["target_met"])),
        "low_recombination_threshold_tree_changes_per_site": (
            low_recombination_threshold
        ),
        "ranked_low_recombination_windows": int(
            ranked_low_recombination_window_indices.size
        ),
        "selected_batch_size": len(selected_local_windows),
        "exact_intervals_at_least_5kb": int(np.sum(_interval_long)),
        "sites_in_exact_intervals_at_least_5kb": {
            "minimum": (
                None
                if _interval_long_site_counts.size == 0
                else int(np.min(_interval_long_site_counts))
            ),
            "median": (
                None
                if _interval_long_site_counts.size == 0
                else float(np.median(_interval_long_site_counts))
            ),
            "maximum": (
                None
                if _interval_long_site_counts.size == 0
                else int(np.max(_interval_long_site_counts))
            ),
        },
        "selected_preview": selected_local_windows[:LOCAL_MAX_DISPLAY],
    }
)


## Local Trace Subgraphs and Fixed Interfaces


In [ ]:
def active_edge_ids_for_tree(tree):
    edge_ids = np.empty(tree.num_edges, dtype=np.int32)
    observed = 0
    for node_id in tree.nodes():
        edge_id = tree.edge(node_id)
        if edge_id != tskit.NULL:
            edge_ids[observed] = edge_id
            observed += 1
    if observed != tree.num_edges:
        raise AssertionError(
            f"tree reports {tree.num_edges} edges but yielded {observed}"
        )
    return edge_ids


def boundary_interface(
    ts,
    local_trace,
    inside_edge_ids,
    outside_tree_index,
    side,
):
    inside_edge_ids = np.unique(
        np.asarray(inside_edge_ids, dtype=np.int32)
    )
    outside_exists = (
        outside_tree_index is not None
        and 0 <= int(outside_tree_index) < ts.num_trees
    )
    if outside_exists:
        outside_tree = ts.at_index(int(outside_tree_index))
        outside_edge_ids = np.unique(active_edge_ids_for_tree(outside_tree))
    else:
        outside_edge_ids = np.empty(0, dtype=np.int32)

    persistent = np.intersect1d(
        inside_edge_ids,
        outside_edge_ids,
        assume_unique=True,
    ).astype(np.int32, copy=False)
    outside_only = np.setdiff1d(
        outside_edge_ids,
        inside_edge_ids,
        assume_unique=True,
    ).astype(np.int32, copy=False)
    inside_only = np.setdiff1d(
        inside_edge_ids,
        outside_edge_ids,
        assume_unique=True,
    ).astype(np.int32, copy=False)

    reconstructed_inside = np.union1d(persistent, inside_only).astype(
        np.int32,
        copy=False,
    )
    reconstructed_outside = np.union1d(persistent, outside_only).astype(
        np.int32,
        copy=False,
    )
    verified = bool(
        np.array_equal(reconstructed_inside, inside_edge_ids)
        and np.array_equal(reconstructed_outside, outside_edge_ids)
    )

    if outside_exists:
        port_edge_ids = np.union1d(inside_edge_ids, outside_edge_ids).astype(
            np.int32,
            copy=False,
        )
        if port_edge_ids.size:
            port_node_ids = np.unique(
                np.concatenate(
                    (
                        local_trace.edge_parent[port_edge_ids],
                        local_trace.edge_child[port_edge_ids],
                    )
                )
            ).astype(np.int32, copy=False)
        else:
            port_node_ids = np.empty(0, dtype=np.int32)
    else:
        port_edge_ids = np.empty(0, dtype=np.int32)
        port_node_ids = np.empty(0, dtype=np.int32)

    return {
        "side": str(side),
        "outside_exists": bool(outside_exists),
        "outside_tree_index": (
            None if not outside_exists else int(outside_tree_index)
        ),
        "inside_edge_count": int(inside_edge_ids.size),
        "outside_edge_count": int(outside_edge_ids.size),
        "persistent_edge_ids": readonly(persistent, np.int32),
        "outside_only_edge_ids": readonly(outside_only, np.int32),
        "inside_only_edge_ids": readonly(inside_only, np.int32),
        "port_edge_ids": readonly(port_edge_ids, np.int32),
        "port_node_ids": readonly(port_node_ids, np.int32),
        "interface_verified": verified,
    }


def rejected_local_subgraph(window, reason):
    empty_i32 = readonly(np.empty(0, dtype=np.int32))
    empty_i64 = readonly(np.empty(0, dtype=np.int64))
    empty_f64 = readonly(np.empty(0, dtype=np.float64))
    return {
        **dict(window),
        "status": "rejected",
        "interface_ready": False,
        "interface_verified": False,
        "has_internal_events": False,
        "rejection_reasons": (str(reason),),
        "local_node_count": 0,
        "local_edge_count": 0,
        "local_event_count": 0,
        "fully_internal_event_count": 0,
        "shared_event_count": 0,
        "local_coalescence_event_count": 0,
        "local_recombination_event_count": 0,
        "local_node_ids": empty_i32,
        "local_edge_ids": empty_i32,
        "local_event_indices": empty_i64,
        "local_event_times": empty_f64,
        "local_event_kinds": empty_i32,
        "fully_internal_event_indices": empty_i64,
        "shared_event_indices": empty_i64,
        "sample_interface_node_ids": empty_i32,
        "local_root_interface_node_ids": empty_i32,
        "fixed_boundary_edge_ids": empty_i32,
        "fixed_boundary_node_ids": empty_i32,
        "fixed_port_node_ids": empty_i32,
        "event_local_edge_offsets": readonly(
            np.zeros(1, dtype=np.int64)
        ),
        "event_local_edge_ids": empty_i32,
        "left_interface": None,
        "right_interface": None,
    }


def extract_local_subgraph(
    ts,
    local_trace,
    window,
    *,
    max_trees,
    max_edge_references,
):
    left = float(window["left"])
    right = float(window["right"])
    start_tree = int(window["start_tree_index"])
    end_tree = int(window["end_tree_index"])
    tree_count = end_tree - start_tree

    if tree_count <= 0:
        return rejected_local_subgraph(window, "empty_tree_range")
    if tree_count > int(max_trees):
        return rejected_local_subgraph(window, "tree_count_guard")

    edge_chunks = []
    root_chunks = []
    tree_root_offsets = np.empty(tree_count + 1, dtype=np.int64)
    tree_root_offsets[0] = 0
    edge_reference_count = 0
    root_count = 0
    first_tree_edges = None
    last_tree_edges = None

    tree = ts.at_index(start_tree)
    for local_tree_index, tree_index in enumerate(
        range(start_tree, end_tree)
    ):
        if tree.index != tree_index:
            raise AssertionError(
                f"expected tree {tree_index}, observed {tree.index}"
            )
        edge_ids = active_edge_ids_for_tree(tree)
        edge_reference_count += int(edge_ids.size)
        if edge_reference_count > int(max_edge_references):
            return rejected_local_subgraph(window, "edge_reference_guard")
        edge_chunks.append(edge_ids)
        roots = np.asarray(tree.roots, dtype=np.int32)
        root_chunks.append(roots)
        root_count += int(roots.size)
        tree_root_offsets[local_tree_index + 1] = root_count
        if local_tree_index == 0:
            first_tree_edges = edge_ids
        last_tree_edges = edge_ids
        if tree_index + 1 < end_tree and not tree.next():
            raise RuntimeError("tree iteration ended inside a window")

    if not edge_chunks:
        return rejected_local_subgraph(window, "no_local_trees")
    local_edge_ids = np.unique(np.concatenate(edge_chunks)).astype(
        np.int32,
        copy=False,
    )
    if local_edge_ids.size == 0:
        return rejected_local_subgraph(window, "no_local_edges")
    tree_root_node_ids = (
        np.concatenate(root_chunks).astype(np.int32, copy=False)
        if root_chunks
        else np.empty(0, dtype=np.int32)
    )
    local_root_node_ids = np.unique(tree_root_node_ids).astype(
        np.int32,
        copy=False,
    )

    original_left = local_trace.edge_left[local_edge_ids]
    original_right = local_trace.edge_right[local_edge_ids]
    clipped_left = np.maximum(original_left, left)
    clipped_right = np.minimum(original_right, right)
    if np.any(clipped_left >= clipped_right):
        raise AssertionError("local edge does not overlap its selected window")

    local_parent = local_trace.edge_parent[local_edge_ids]
    local_child = local_trace.edge_child[local_edge_ids]
    local_node_ids = np.unique(
        np.concatenate(
            (
                local_parent,
                local_child,
                local_root_node_ids,
            )
        )
    ).astype(np.int32, copy=False)

    reveal_step = local_trace.node_reveal_step[local_parent]
    if np.any(reveal_step <= 0):
        bad_parents = local_parent[reveal_step <= 0]
        raise RuntimeError(
            f"local edge parent has no creating event: {bad_parents[:10]}"
        )
    edge_event_indices = reveal_step.astype(np.int64) - 1
    event_edge_order = np.lexsort((local_edge_ids, edge_event_indices))
    event_local_edge_ids = local_edge_ids[event_edge_order]
    sorted_edge_events = edge_event_indices[event_edge_order]
    local_event_indices, event_first, event_counts = np.unique(
        sorted_edge_events,
        return_index=True,
        return_counts=True,
    )
    event_local_edge_offsets = np.empty(
        local_event_indices.size + 1,
        dtype=np.int64,
    )
    event_local_edge_offsets[0] = 0
    np.cumsum(event_counts, out=event_local_edge_offsets[1:])

    fully_internal_mask = np.empty(local_event_indices.size, dtype=np.bool_)
    for event_offset, event_index in enumerate(local_event_indices):
        event_index = int(event_index)
        edge_start = int(local_trace.event_edge_start[event_index])
        edge_end = int(local_trace.event_edge_start[event_index + 1])
        full_event_edge_ids = local_trace.revealed_edge_ids[
            edge_start:edge_end
        ]
        fully_internal_mask[event_offset] = bool(
            full_event_edge_ids.size
            and np.all(local_trace.edge_left[full_event_edge_ids] >= left)
            and np.all(local_trace.edge_right[full_event_edge_ids] <= right)
        )

    fully_internal_event_indices = local_event_indices[
        fully_internal_mask
    ].astype(np.int64, copy=False)
    shared_event_indices = local_event_indices[
        ~fully_internal_mask
    ].astype(np.int64, copy=False)
    local_event_kinds = local_trace.event_kind[local_event_indices]
    local_event_times = local_trace.event_time[local_event_indices]
    sample_interface_node_ids = np.intersect1d(
        local_node_ids,
        local_trace.sample_nodes,
        assume_unique=True,
    ).astype(np.int32, copy=False)

    left_interface = boundary_interface(
        ts,
        local_trace,
        first_tree_edges,
        start_tree - 1 if start_tree > 0 else None,
        "left",
    )
    right_interface = boundary_interface(
        ts,
        local_trace,
        last_tree_edges,
        end_tree if end_tree < ts.num_trees else None,
        "right",
    )
    fixed_boundary_edge_ids = np.union1d(
        left_interface["port_edge_ids"],
        right_interface["port_edge_ids"],
    ).astype(np.int32, copy=False)
    fixed_boundary_node_ids = np.union1d(
        left_interface["port_node_ids"],
        right_interface["port_node_ids"],
    ).astype(np.int32, copy=False)
    fixed_port_node_ids = np.unique(
        np.concatenate(
            (
                fixed_boundary_node_ids,
                sample_interface_node_ids,
                local_root_node_ids,
            )
        )
    ).astype(np.int32, copy=False)
    interface_verified = bool(
        left_interface["interface_verified"]
        and right_interface["interface_verified"]
    )
    rejection_reasons = () if interface_verified else (
        "boundary_interface_not_verified",
    )

    return {
        **dict(window),
        "status": "ok" if interface_verified else "rejected",
        "interface_ready": interface_verified,
        "interface_verified": interface_verified,
        "rejection_reasons": rejection_reasons,
        "sample_node_ids": local_trace.sample_nodes,
        "sample_count": int(local_trace.sample_nodes.size),
        "local_node_ids": readonly(local_node_ids, np.int32),
        "local_edge_ids": readonly(local_edge_ids, np.int32),
        "local_event_indices": readonly(local_event_indices, np.int64),
        "local_event_times": readonly(local_event_times, np.float64),
        "local_event_kinds": readonly(local_event_kinds),
        "local_node_count": int(local_node_ids.size),
        "local_edge_count": int(local_edge_ids.size),
        "local_event_count": int(local_event_indices.size),
        "local_coalescence_event_count": int(
            np.sum(local_event_kinds == EVENT_KIND_COALESCENCE)
        ),
        "local_recombination_event_count": int(
            np.sum(local_event_kinds == EVENT_KIND_RECOMBINATION)
        ),
        "fully_internal_event_indices": readonly(
            fully_internal_event_indices,
            np.int64,
        ),
        "shared_event_indices": readonly(shared_event_indices, np.int64),
        "fully_internal_event_count": int(
            fully_internal_event_indices.size
        ),
        "shared_event_count": int(shared_event_indices.size),
        "has_internal_events": bool(fully_internal_event_indices.size),
        "event_local_edge_offsets": readonly(
            event_local_edge_offsets,
            np.int64,
        ),
        "event_local_edge_ids": readonly(event_local_edge_ids, np.int32),
        "edge_reference_count": int(edge_reference_count),
        "tree_root_offsets": readonly(tree_root_offsets, np.int64),
        "tree_root_node_ids": readonly(tree_root_node_ids, np.int32),
        "local_root_node_ids": readonly(local_root_node_ids, np.int32),
        "sample_interface_node_ids": readonly(
            sample_interface_node_ids,
            np.int32,
        ),
        "local_root_interface_node_ids": readonly(
            local_root_node_ids,
            np.int32,
        ),
        "fixed_boundary_edge_ids": readonly(
            fixed_boundary_edge_ids,
            np.int32,
        ),
        "fixed_boundary_node_ids": readonly(
            fixed_boundary_node_ids,
            np.int32,
        ),
        "fixed_port_node_ids": readonly(fixed_port_node_ids, np.int32),
        "left_interface": left_interface,
        "right_interface": right_interface,
        "local_edge_slices": {
            "edge_id": readonly(local_edge_ids, np.int32),
            "left": readonly(clipped_left, np.float64),
            "right": readonly(clipped_right, np.float64),
            "parent": readonly(local_parent, np.int32),
            "child": readonly(local_child, np.int32),
        },
    }


def local_subgraph_preview(candidate):
    return {
        "window_id": candidate["window_id"],
        "selection_rank": candidate.get("selection_rank"),
        "interval": (candidate["left"], candidate["right"]),
        "span": candidate["span"],
        "sites": candidate["site_count"],
        "trees": candidate["tree_count"],
        "tree_changes_per_site": candidate["tree_changes_per_site"],
        "status": candidate["status"],
        "interface_ready": candidate["interface_ready"],
        "nodes": candidate["local_node_count"],
        "edges": candidate["local_edge_count"],
        "events": candidate["local_event_count"],
        "coalescence_events": candidate.get(
            "local_coalescence_event_count",
            0,
        ),
        "recombination_events": candidate.get(
            "local_recombination_event_count",
            0,
        ),
        "fully_internal_events": candidate["fully_internal_event_count"],
        "shared_events": candidate["shared_event_count"],
        "rejection_reasons": candidate["rejection_reasons"],
    }


## Synthetic Helper Checks


In [ ]:
def build_spatial_test_arg():
    tables = tskit.TableCollection(sequence_length=30.0)
    samples = [
        tables.nodes.add_row(flags=tskit.NODE_IS_SAMPLE, time=0.0)
        for _ in range(4)
    ]

    shared_parent = tables.nodes.add_row(flags=0, time=1.0)
    tables.edges.add_row(0.0, 30.0, parent=shared_parent, child=samples[0])
    tables.edges.add_row(0.0, 30.0, parent=shared_parent, child=samples[1])

    def add_local_tree(
        left,
        right,
        ingroup_sample,
        outgroup_sample,
        ingroup_time,
        root_time,
    ):
        ingroup = tables.nodes.add_row(flags=0, time=ingroup_time)
        root = tables.nodes.add_row(flags=0, time=root_time)
        tables.edges.add_row(left, right, parent=ingroup, child=shared_parent)
        tables.edges.add_row(left, right, parent=ingroup, child=ingroup_sample)
        tables.edges.add_row(left, right, parent=root, child=ingroup)
        tables.edges.add_row(left, right, parent=root, child=outgroup_sample)

    add_local_tree(0.0, 10.0, samples[2], samples[3], 1.2, 2.0)
    add_local_tree(10.0, 20.0, samples[3], samples[2], 1.4, 2.2)
    add_local_tree(20.0, 30.0, samples[2], samples[3], 1.6, 2.4)

    for position in (1.0, 3.0, 11.0, 13.0, 21.0, 23.0):
        site = tables.sites.add_row(position=position, ancestral_state="0")
        tables.mutations.add_row(site=site, node=samples[0], derived_state="1")

    tables.sort()
    source = tables.tree_sequence()
    return build_synthetic_full_arg(
        source,
        split_rule="balanced",
        ensure_unique_event_times=True,
    ).tree_sequence


_spatial_test_arg = build_spatial_test_arg()
_spatial_test_source_signature = tree_sequence_signature(
    _spatial_test_arg
)
_spatial_test_trace = build_fast_trace_from_full_arg(
    _spatial_test_arg,
    require_unique_event_times=True,
)
_spatial_test_intervals = build_tree_interval_catalog(_spatial_test_arg)
assert _spatial_test_intervals["left"].size == 3
assert np.array_equal(
    _spatial_test_intervals["breakpoints"],
    np.array([0.0, 10.0, 20.0, 30.0]),
)
assert int(np.sum(_spatial_test_intervals["site_count"])) == 6
assert int(np.sum(_spatial_test_intervals["mutation_count"])) == 6

_spatial_test_windows = build_local_window_catalog(
    _spatial_test_intervals,
    4,
)
assert _spatial_test_windows["left"].tolist() == [0.0, 20.0]
assert _spatial_test_windows["right"].tolist() == [20.0, 30.0]
assert _spatial_test_windows["site_count"].tolist() == [4, 2]
assert _spatial_test_windows["tree_count"].tolist() == [2, 1]

_spatial_test_multi = window_record(_spatial_test_windows, 0, 0)
_spatial_test_candidate = extract_local_subgraph(
    _spatial_test_arg,
    _spatial_test_trace,
    _spatial_test_multi,
    max_trees=10,
    max_edge_references=10000,
)
assert _spatial_test_candidate["interface_ready"]
assert not _spatial_test_candidate["left_interface"]["outside_exists"]
assert _spatial_test_candidate["right_interface"]["outside_exists"]
assert (
    _spatial_test_candidate["fully_internal_event_count"]
    + _spatial_test_candidate["shared_event_count"]
    == _spatial_test_candidate["local_event_count"]
)
assert _spatial_test_candidate["fully_internal_event_count"] > 0
assert _spatial_test_candidate["shared_event_count"] > 0
assert set(_spatial_test_candidate["sample_interface_node_ids"]) == set(
    _spatial_test_trace.sample_nodes
)
assert set(_spatial_test_candidate["sample_interface_node_ids"]).issubset(
    set(_spatial_test_candidate["fixed_port_node_ids"])
)
assert np.all(
    _spatial_test_candidate["local_edge_slices"]["left"]
    >= _spatial_test_candidate["left"]
)
assert np.all(
    _spatial_test_candidate["local_edge_slices"]["right"]
    <= _spatial_test_candidate["right"]
)
assert (
    _spatial_test_candidate["event_local_edge_offsets"][-1]
    == _spatial_test_candidate["local_edge_count"]
)

_spatial_test_single = {
    "window_id": "fixture-single-tree",
    "window_index": 2,
    "window_key": (20.0, 30.0),
    "left": 20.0,
    "right": 30.0,
    "span": 10.0,
    "site_count": 2,
    "mutation_count": 2,
    "start_tree_index": 2,
    "end_tree_index": 3,
    "tree_count": 1,
    "tree_change_count": 0,
    "tree_changes_per_site": 0.0,
    "target_met": False,
    "selection_rank": 1,
}
_spatial_test_single_candidate = extract_local_subgraph(
    _spatial_test_arg,
    _spatial_test_trace,
    _spatial_test_single,
    max_trees=10,
    max_edge_references=10000,
)
assert _spatial_test_single_candidate["interface_ready"]
assert _spatial_test_single_candidate["left_interface"]["outside_exists"]
assert not _spatial_test_single_candidate["right_interface"]["outside_exists"]
assert _spatial_test_single_candidate["tree_count"] == 1

assert _spatial_test_source_signature == tree_sequence_signature(
    _spatial_test_arg
)

helper_validation_summary = {
    "exact_single_tree_block": "passed",
    "multi_tree_site_window": "passed",
    "chromosome_end_interfaces": "passed",
    "shared_internal_event_partition": "passed",
    "boundary_difference_reconstruction": "passed",
    "source_immutability": "passed",
}
display(helper_validation_summary)

del _spatial_test_arg
del _spatial_test_trace
gc.collect()


## Deep-Trace the Selected Low-Recombination Batch


In [ ]:
def extract_selected_local_subgraphs():
    results = []
    extraction_order = sorted(
        selected_local_windows,
        key=lambda window: (
            window["start_tree_index"],
            window["end_tree_index"],
        ),
    )
    started = time.perf_counter()
    for index, window in enumerate(extraction_order, start=1):
        candidate = extract_local_subgraph(
            synthetic_arg,
            trace,
            window,
            max_trees=LOCAL_MAX_TREES_PER_WINDOW,
            max_edge_references=LOCAL_MAX_EDGE_REFERENCES,
        )
        results.append(candidate)
        if index == 1 or index % 10 == 0 or index == len(extraction_order):
            elapsed = time.perf_counter() - started
            emit(
                "Local subgraph extraction: "
                f"{index}/{len(extraction_order)} windows | "
                f"elapsed={elapsed:.1f}s | "
                f"max_rss={max_rss_gib():.2f} GiB"
            )
    results.sort(key=lambda candidate: candidate.get("selection_rank", 0))
    return results


local_subgraph_candidates = run_stage(
    "Extract selected local trace subgraphs",
    extract_selected_local_subgraphs,
)
interface_ready_local_subgraphs = [
    candidate
    for candidate in local_subgraph_candidates
    if candidate["interface_ready"]
]
subgraphs_with_internal_events = [
    candidate
    for candidate in interface_ready_local_subgraphs
    if candidate["has_internal_events"]
]

_preview_count = min(LOCAL_MAX_DISPLAY, len(local_subgraph_candidates))
display(
    {
        "selected_windows": len(selected_local_windows),
        "local_subgraph_candidates": len(local_subgraph_candidates),
        "interface_ready": len(interface_ready_local_subgraphs),
        "with_fully_internal_events": len(subgraphs_with_internal_events),
        "displayed": _preview_count,
        "candidates": [
            local_subgraph_preview(candidate)
            for candidate in local_subgraph_candidates[:_preview_count]
        ],
    }
)


## Live-Data Validation


In [ ]:
def assert_partition(left, right, sequence_length):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)
    assert left.size == right.size and left.size > 0
    assert left[0] == 0.0
    assert right[-1] == float(sequence_length)
    assert np.all(left < right)
    assert np.array_equal(right[:-1], left[1:])


assert_partition(
    tree_interval_catalog["left"],
    tree_interval_catalog["right"],
    synthetic_arg.sequence_length,
)
assert_partition(
    local_window_catalog["left"],
    local_window_catalog["right"],
    synthetic_arg.sequence_length,
)
assert tree_interval_catalog["left"].size == synthetic_arg.num_trees
assert int(np.sum(tree_interval_catalog["site_count"])) == synthetic_arg.num_sites
assert (
    int(np.sum(tree_interval_catalog["mutation_count"]))
    == synthetic_arg.num_mutations
)
assert int(np.sum(local_window_catalog["site_count"])) == synthetic_arg.num_sites
assert (
    int(np.sum(local_window_catalog["mutation_count"]))
    == synthetic_arg.num_mutations
)

_selected_indices = np.array(
    [window["window_index"] for window in selected_local_windows],
    dtype=np.int64,
)
if _selected_indices.size:
    assert np.all(local_window_catalog["target_met"][_selected_indices])
    assert np.all(
        local_window_catalog["tree_changes_per_site"][_selected_indices]
        <= low_recombination_threshold
    )

assert len(local_subgraph_candidates) == len(selected_local_windows)
for candidate in interface_ready_local_subgraphs:
    assert candidate["status"] == "ok"
    assert candidate["interface_verified"]
    assert candidate["left_interface"]["interface_verified"]
    assert candidate["right_interface"]["interface_verified"]
    assert candidate["local_event_count"] == (
        candidate["fully_internal_event_count"]
        + candidate["shared_event_count"]
    )
    assert candidate["event_local_edge_offsets"].shape == (
        candidate["local_event_count"] + 1,
    )
    assert int(candidate["event_local_edge_offsets"][-1]) == (
        candidate["local_edge_count"]
    )
    assert np.array_equal(
        np.sort(candidate["event_local_edge_ids"]),
        candidate["local_edge_ids"],
    )
    assert candidate["local_event_times"].shape == (
        candidate["local_event_count"],
    )
    assert candidate["local_event_kinds"].shape == (
        candidate["local_event_count"],
    )
    assert np.array_equal(
        candidate["local_event_times"],
        trace.event_time[candidate["local_event_indices"]],
    )
    assert np.array_equal(
        candidate["local_event_kinds"],
        trace.event_kind[candidate["local_event_indices"]],
    )
    slices = candidate["local_edge_slices"]
    assert np.all(slices["left"] >= candidate["left"])
    assert np.all(slices["right"] <= candidate["right"])
    assert np.all(slices["left"] < slices["right"])
    assert np.all(candidate["local_node_ids"] >= 0)
    assert np.all(candidate["local_node_ids"] < trace.node_time.size)
    assert np.all(candidate["local_event_indices"] >= 0)
    assert np.all(candidate["local_event_indices"] < trace.event_count)
    assert np.all(candidate["sample_interface_node_ids"] >= 0)
    assert np.all(
        candidate["sample_interface_node_ids"] < trace.node_time.size
    )
    assert set(candidate["sample_interface_node_ids"]).issubset(
        set(candidate["fixed_port_node_ids"])
    )
    assert np.all(candidate["fixed_boundary_edge_ids"] >= 0)
    assert np.all(
        candidate["fixed_boundary_edge_ids"] < trace.edge_parent.size
    )
    assert np.all(candidate["fixed_boundary_node_ids"] >= 0)
    assert np.all(
        candidate["fixed_boundary_node_ids"] < trace.node_time.size
    )
    assert np.all(candidate["fixed_port_node_ids"] >= 0)
    assert np.all(
        candidate["fixed_port_node_ids"] < trace.node_time.size
    )
    if candidate["local_event_indices"].size > 1:
        assert np.all(np.diff(candidate["local_event_indices"]) > 0)

assert synthetic_signature_before == tree_sequence_signature(synthetic_arg)
assert trace_signature_before == {
    "steps": int(trace.num_steps),
    "events": int(trace.event_count),
    "nodes": int(trace.node_time.size),
    "edges": int(trace.edge_parent.size),
    "samples": int(trace.sample_nodes.size),
    "event_times_strictly_increasing": bool(
        np.all(np.diff(trace.event_time) > 0.0)
    ),
    "adjacent_tied_event_times": int(
        np.sum(np.diff(trace.event_time) == 0.0)
    ),
}

validation_summary = {
    "tree_intervals_partition_sequence": True,
    "site_windows_partition_sequence": True,
    "site_and_mutation_counts_preserved": True,
    "selected_windows_inside_low_recombination_quantile": True,
    "local_edges_map_to_one_temporal_event": True,
    "boundary_interfaces_reconstruct_both_sides": True,
    "edge_slices_stay_inside_windows": True,
    "synthetic_arg_unchanged": True,
    "trace_unchanged": True,
}
display(validation_summary)


## Live Objects

- `synthetic_arg` is the immutable synthetic full ARG loaded from the prebuilt tree sequence.
- `trace` is its immutable `FastARGTrace`; this notebook does not construct a terminal cursor.
- `tree_interval_catalog` is a columnar catalog of every exact marginal-tree interval.
- `local_window_catalog` partitions the chromosome into tree-aligned windows targeting 100 sites.
- `selected_local_windows` contains the ranked low-recombination batch chosen for deep tracing.
- `local_subgraph_candidates` contains original clipped nodes, edges, events, roots, and boundary interfaces for every selected window.
- `interface_ready_local_subgraphs` contains candidates whose left and right fixed interfaces were reconstructed and verified.
- `subgraphs_with_internal_events` contains interface-ready candidates with at least one fully internal original event.

These objects describe possible future update domains. They do not claim that a region is bad, and they do not apply or score replacement actions.
